### Human small run: Stage 1 SSL (causal AR) on 4 B2T sessions
- Runs `scripts/pretrain_ssl.py` with `configs/ssl_human.yaml`; outputs go to `experiments/human_small_run/`
- Data: B2T '25 copy task, sessions t15.2023.08.13 / 08.18 / 08.20 / 08.25 (512 features per 20 ms bin)
- Objective: from each 14-bin patch (stride 4 = 80 ms), predict the next 4 bins of z-scored features; MSE loss
- Budget: 300 steps (1.4-2.0 s/step on MPS, 5 min total), warm-up 30, cosine decay over 300, val every 50 steps on 12 val batches
- Baselines on the same val crops: `zero` (predict the mean, i.e. 0 for z-scored features) and `persistence` (repeat the last observed bin)

In [1]:
import json, os, re, subprocess, sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
RUN_NAME = 'human_small_run'
RUN_DIR = PROJECT_ROOT / 'experiments' / RUN_NAME
RESULTS_CSV = RUN_DIR / 'results.csv'
RERUN = False  # True: train again even if a finished run is on disk

SESSIONS = ['t15.2023.08.13', 't15.2023.08.18', 't15.2023.08.20', 't15.2023.08.25']
OVERRIDES = [
    f"b2t_sessions=[{','.join(SESSIONS)}]",
    'num_training_batches=300',
    'lr_warmup_steps=30',
    'lr_decay_steps=300',
    'batches_per_train_log=10',
    'batches_per_val_step=50',
]

In [2]:
# training runs as a subprocess: DataLoader workers need the script's __main__ guard (spawn on macOS)
if RERUN or not (RUN_DIR / 'checkpoint' / 'final_checkpoint').exists():
    cmd = [sys.executable, 'scripts/pretrain_ssl.py', '--config', 'configs/ssl_human.yaml',
           '--run_name', RUN_NAME, *OVERRIDES]
    print(' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    assert proc.wait() == 0, f'training failed (exit {proc.returncode})'
else:
    print(f'finished run found in {RUN_DIR}, skipping training')

/Users/cmoonk/Projects/nejm-brain-to-text-rp/prt-rpn-karya/.venv-train/bin/python scripts/pretrain_ssl.py --config configs/ssl_human.yaml --run_name human_small_run b2t_sessions=[t15.2023.08.13,t15.2023.08.18,t15.2023.08.20,t15.2023.08.25] num_training_batches=300 lr_warmup_steps=30 lr_decay_steps=300 batches_per_train_log=10 batches_per_val_step=50


2026-10-29 18:01:43,397: Using device: mps


2026-10-29 18:01:43,401: source b2t: 4 sessions, channels [512], weight 1.0, loss linear


2026-10-29 18:01:43,557: SpikeGPT backbone: 11,528,448 params | stems + AR heads: 3,054,336 params


2026-10-29 18:01:48,780: Train batch 0 [b2t]: loss 1.3368 grad norm 0.07 time 1.232


2026-10-29 18:01:49,998: Val batch 0 [b2t]: loss 1.3411 zero 1.0086 persistence 1.7521


... (47 outputs left out)

2026-10-29 18:06:22,167: Val batch 299: weighted loss 0.9585 time 1.8s


2026-10-29 18:06:22,255: Saved checkpoint: experiments/human_small_run/checkpoint/best_checkpoint


2026-10-29 18:06:42,457: Saved checkpoint: experiments/human_small_run/checkpoint/final_checkpoint


2026-10-29 18:06:42,457: Best weighted val loss 0.9585, total time 5.0 min


In [3]:
# one row per logged step: train loss every batches_per_train_log steps, val metrics every batches_per_val_step
train_re = re.compile(r'Train batch (\d+) \[(\w+)\]: loss ([\d.]+) grad norm ([\d.]+)')
train = pd.DataFrame(
    [(int(m[1]), float(m[3]), float(m[4])) for m in map(train_re.search, open(RUN_DIR / 'training_log')) if m],
    columns=['step', 'train_loss', 'grad_norm'],
).drop_duplicates('step', keep='last')  # a rerun appends to the same log

val = pd.DataFrame([
    {'step': h['step'], **{f'val_{k}': v for k, v in h['val']['b2t'].items()}}
    for h in json.load(open(RUN_DIR / 'val_history.json'))
])

results = train.merge(val, on='step', how='outer').sort_values('step')
results.to_csv(RESULTS_CSV, index=False)
print(f'saved {len(results)} rows to {RESULTS_CSV}')

saved 31 rows to /Users/cmoonk/Projects/nejm-brain-to-text-rp/prt-rpn-karya/experiments/human_small_run/results.csv


### Results (read back from `results.csv`)
- `train_loss`: MSE on one augmented training batch (noisy, single batch)
- `val_loss`: MSE on the fixed val crops; `val_zero` and `val_persistence` are the baselines on the same crops
- A useful model has `val_loss` below both baselines

In [4]:
res = pd.read_csv(RESULTS_CSV)
v = res.dropna(subset=['val_loss'])
best = v.loc[v['val_loss'].idxmin()]
print(f"best val loss {best['val_loss']:.4f} at step {int(best['step'])} | "
      f"zero {best['val_zero']:.4f} ({100 * (1 - best['val_loss'] / best['val_zero']):.1f}% below) | "
      f"persistence {best['val_persistence']:.4f} ({100 * (1 - best['val_loss'] / best['val_persistence']):.1f}% below)")
display(v[['step', 'train_loss', 'val_loss', 'val_zero', 'val_persistence']].round(4).reset_index(drop=True))

t = res.dropna(subset=['train_loss'])
fig = go.Figure()
fig.add_trace(go.Scatter(x=t['step'], y=t['train_loss'], mode='lines', name='train loss',
                         line=dict(color='lightgrey')))
fig.add_trace(go.Scatter(x=v['step'], y=v['val_loss'], mode='lines+markers', name='val loss',
                         line=dict(color='#1f77b4', width=2)))
fig.add_trace(go.Scatter(x=v['step'], y=v['val_zero'], mode='lines', name='baseline: zero',
                         line=dict(color='black', dash='dot')))
fig.add_trace(go.Scatter(x=v['step'], y=v['val_persistence'], mode='lines', name='baseline: persistence',
                         line=dict(color='#d62728', dash='dash')))
fig.update_layout(
    template='plotly_white', height=450, hovermode='x unified',
    title=f'{RUN_NAME}: Stage 1 causal AR SSL, {len(SESSIONS)} B2T sessions',
    xaxis_title='Training step (batch)', yaxis_title='MSE, next 4 bins (z-scored features)',
)
fig.show()

best val loss 0.9585 at step 299 | zero 1.0086 (5.0% below) | persistence 1.7521 (45.3% below)


,step,train_loss,val_loss,val_zero,val_persistence
0,0,1.3368,1.3411,1.0086,1.7521
1,50,1.0180,0.9856,1.0086,1.7521
2,100,1.0000,0.9715,1.0086,1.7521
3,150,0.9886,0.9645,1.0086,1.7521
4,200,0.9826,0.9608,1.0086,1.7521
5,250,0.9910,0.9590,1.0086,1.7521
6,299,NaN,0.9585,1.0086,1.7521


### Training loss curve
- `train_loss` is logged every 10 steps, on one batch of noise-augmented inputs (white noise 0.5, offset 0.2); the target is the clean features
- Rolling mean over 5 logged points (50 steps) smooths the batch-to-batch noise
- Grad norm (right axis) is before clipping at 1.0
- Dotted line: `zero` baseline on val, i.e. the MSE of always predicting the per-feature mean

In [5]:
t = res.dropna(subset=['train_loss'])
zero = res['val_zero'].dropna().iloc[0]

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=t['step'], y=t['train_loss'], mode='lines+markers', name='train loss',
                         line=dict(color='#1f77b4', width=1), marker=dict(size=5)), secondary_y=False)
fig.add_trace(go.Scatter(x=t['step'], y=t['train_loss'].rolling(5, min_periods=1).mean(), mode='lines',
                         name='train loss, rolling mean (5)', line=dict(color='#1f77b4', width=3)), secondary_y=False)
fig.add_trace(go.Scatter(x=t['step'], y=t['grad_norm'], mode='lines', name='grad norm',
                         line=dict(color='#ff7f0e', width=1, dash='dash')), secondary_y=True)
fig.add_hline(y=zero, line=dict(color='black', dash='dot'), annotation_text=f'zero baseline (val) {zero:.3f}',
              annotation_position='top right')
fig.update_layout(
    template='plotly_white', height=450, hovermode='x unified',
    title=f'{RUN_NAME}: training loss',
    xaxis_title='Training step (batch)',
)
fig.update_yaxes(title_text='Train MSE, next 4 bins (z-scored features)', secondary_y=False)
fig.update_yaxes(title_text='Grad norm (pre-clip)', secondary_y=True, showgrid=False)
fig.show()

### Stage 2: phoneme CTC fine-tuning, SSL init vs random init
- Runs `scripts/finetune_ctc.py` with `configs/ctc_human.yaml` on the same 4 sessions as Stage 1 (911 train / 157 val trials), so the Stage 1 stem (per-day input layers) is reused
- `ssl`: backbone + stem from `human_small_run/checkpoint/best_checkpoint`; `random`: same architecture, no pretraining (control)
- Same schedule for both: 500 steps, backbone frozen for the first 100 (stem + CTC head warm up), then everything trains (backbone LR 3e-4, rest 2e-3)
- Metric: phoneme error rate (PER) with greedy CTC decoding on the 157 val trials; val every 50 steps
- Outputs: `experiments/human_small_run/ctc_ssl/` and `ctc_random/`; the best-PER checkpoint is kept

In [6]:
import pickle, shutil

CTC_STEPS, CTC_FREEZE = 500, 100
TRAIN_CTC = False  # False: only use runs already on disk (ssl was stopped at step 480; random was never run)
CTC_RUNS = {
    'ssl': ['init=ssl', f'init_checkpoint=experiments/{RUN_NAME}/checkpoint/best_checkpoint'],
    'random': ['init=random'],
}
CTC_OVERRIDES = [
    f"b2t_sessions=[{','.join(SESSIONS)}]",
    f'num_training_batches={CTC_STEPS}',
    f'freeze_backbone_steps={CTC_FREEZE}',
    'lr_warmup_steps=50',
    f'lr_decay_steps={CTC_STEPS}',
    'batches_per_train_log=10',
    'batches_per_val_step=50',
]

def ctc_done(run_dir):
    log = run_dir / 'training_log'
    return log.exists() and 'Total training time' in log.read_text()

for name, init in CTC_RUNS.items():
    run_dir = RUN_DIR / f'ctc_{name}'
    if run_dir.exists() and not RERUN:
        print(f"{'finished' if ctc_done(run_dir) else 'partial'} run found in {run_dir}, skipping training")
        continue
    if not TRAIN_CTC:
        print(f'ctc_{name}: not on disk, TRAIN_CTC is False, skipping')
        continue
    if run_dir.exists():  # RERUN: the trainer refuses an existing output_dir
        shutil.rmtree(run_dir)
    cmd = [sys.executable, 'scripts/finetune_ctc.py', '--config', 'configs/ctc_human.yaml',
           '--run_name', f'{RUN_NAME}/ctc_{name}', *init, *CTC_OVERRIDES]
    print(' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    assert proc.wait() == 0, f'ctc_{name} failed (exit {proc.returncode})'

partial run found in /Users/cmoonk/Projects/nejm-brain-to-text-rp/prt-rpn-karya/experiments/human_small_run/ctc_ssl, skipping training
ctc_random: not on disk, TRAIN_CTC is False, skipping


In [7]:
# ctc_results.csv: one row per (run, logged step); ctc_day_PER.csv: per-session val PER of each run's best checkpoint
CTC_CSV = RUN_DIR / 'ctc_results.csv'
CTC_DAY_CSV = RUN_DIR / 'ctc_day_PER.csv'
ctc_train_re = re.compile(r'Train batch (\d+): loss ([\d.]+) grad norm ([\d.]+)')

rows, day_rows = [], []
for name in CTC_RUNS:
    run_dir = RUN_DIR / f'ctc_{name}'
    if not (run_dir / 'val_history.json').exists():
        print(f'ctc_{name}: no results on disk, left out')
        continue
    tr = pd.DataFrame(
        [(int(m[1]), float(m[2]), float(m[3])) for m in map(ctc_train_re.search, open(run_dir / 'training_log')) if m],
        columns=['step', 'train_loss', 'grad_norm'],
    )
    va = pd.DataFrame(json.load(open(run_dir / 'val_history.json')))[['step', 'PER', 'loss']]
    va.columns = ['step', 'val_PER', 'val_loss']
    rows.append(tr.merge(va, on='step', how='outer').assign(run=name))
    best = pickle.load(open(run_dir / 'checkpoint' / 'val_metrics.pkl', 'rb'))
    day_rows += [{'run': name, 'session': s, 'val_PER': p} for s, p in best['day_PERs'].items()]

ctc = pd.concat(rows)[['run', 'step', 'train_loss', 'grad_norm', 'val_loss', 'val_PER']].sort_values(['run', 'step'])
ctc.to_csv(CTC_CSV, index=False)
pd.DataFrame(day_rows).to_csv(CTC_DAY_CSV, index=False)
print(f'saved {len(ctc)} rows to {CTC_CSV}\nsaved {len(day_rows)} rows to {CTC_DAY_CSV}')

ctc_random: no results on disk, left out
saved 49 rows to /Users/cmoonk/Projects/nejm-brain-to-text-rp/prt-rpn-karya/experiments/human_small_run/ctc_results.csv
saved 4 rows to /Users/cmoonk/Projects/nejm-brain-to-text-rp/prt-rpn-karya/experiments/human_small_run/ctc_day_PER.csv


### Stage 2 results (read back from `ctc_results.csv` and `ctc_day_PER.csv`)
- PER = phoneme edit distance / number of target phonemes, over all 157 val trials
- Grey band: steps where the backbone is frozen
- Per-session PER is from each run's best checkpoint

In [8]:
ctc = pd.read_csv(CTC_CSV)
day = pd.read_csv(CTC_DAY_CSV)
COLORS = {'ssl': '#1f77b4', 'random': '#7f7f7f'}

v = ctc.dropna(subset=['val_PER'])
summary = v.loc[v.groupby('run')['val_PER'].idxmin(), ['run', 'step', 'val_PER', 'val_loss']]
summary.columns = ['run', 'best step', 'best val PER', 'val CTC loss at best']
display(summary.round(4).reset_index(drop=True))
display(day.pivot(index='session', columns='run', values='val_PER').round(4))

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=('Validation phoneme error rate', 'CTC loss'))
for name, d in ctc.groupby('run'):
    dv, dt = d.dropna(subset=['val_PER']), d.dropna(subset=['train_loss'])
    fig.add_trace(go.Scatter(x=dv['step'], y=100 * dv['val_PER'], mode='lines+markers', name=f'{name}: val PER',
                             line=dict(color=COLORS[name], width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=dt['step'], y=dt['train_loss'], mode='lines', name=f'{name}: train loss',
                             line=dict(color=COLORS[name], width=1, dash='dot')), row=1, col=2)
    fig.add_trace(go.Scatter(x=dv['step'], y=dv['val_loss'], mode='lines+markers', name=f'{name}: val loss',
                             line=dict(color=COLORS[name], width=2)), row=1, col=2)
for c in (1, 2):
    fig.add_vrect(x0=0, x1=CTC_FREEZE, fillcolor='lightgrey', opacity=0.3, line_width=0, row=1, col=c)
    fig.update_xaxes(title_text='Training step (batch)', row=1, col=c)
fig.update_yaxes(title_text='PER (%)', row=1, col=1)
fig.update_yaxes(title_text='CTC loss', row=1, col=2)
fig.update_layout(template='plotly_white', height=480, hovermode='x unified',
                  title=f'{RUN_NAME}: Stage 2 CTC, SSL init vs random init ({len(SESSIONS)} sessions)')
fig.show()

,run,best step,best val PER,val CTC loss at best
0,ssl,450,0.4963,48.6757


run,ssl
session,
t15.2023.08.13,0.4958
t15.2023.08.18,0.5038
t15.2023.08.20,0.4956
t15.2023.08.25,0.4849
